In [0]:
# Databricks notebook source
# src/pipelines/gold_pipeline.py

import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =========================================================================
# 1. GOLD CONFORMED DIMENSIONS
# =========================================================================

@dlt.table(
    name="gold.dim_users",
    comment="Conformed Users SCD Type 2 dimension. Grain: one row per user version."
)
def gold_dim_users():
    return (
        dlt.read("silver.dim_users")
           .select(
               F.col("user_id").alias("user_id"),
               F.col("name").alias("user_name"),
               F.col("email").alias("email"),
               F.col("country").alias("country"),
               F.col("user_type").alias("user_type"),
               F.col("is_business").alias("is_business"),
               F.col("company_name").alias("company_name"),
               F.col("__START_AT").alias("valid_from"),
               F.col("__END_AT").alias("valid_to")
           )
    )

@dlt.table(
    name="gold.dim_properties",
    comment="Conformed Properties dimension. Grain: one row per property listing."
)
def gold_dim_properties():
    return (
        dlt.read("silver.dim_properties")
           .select(
               "property_id",
               "host_id",
               "destination_id",
               "title",
               "description",
               "base_price",
               "property_type",
               "max_guests",
               "bedrooms",
               "bathrooms",
               "property_latitude",
               "property_longitude"
           )
    )

# =========================================================================
# 2. TEMPORARY PIT EXPANSION FOR RETROACTIVE HISTORICAL TRANSACTIONS
# =========================================================================

@dlt.table(
    name="users_pit_staging",
    comment="Internal staging view with beginning-of-time interval expansion for earliest version",
    temporary=True
)
def users_pit_staging():
    users = dlt.read("silver.dim_users")
    w = Window.partitionBy("user_id").orderBy(F.col("__START_AT").asc())
    
    return (
        users.withColumn("rn", F.row_number().over(w))
             .withColumn(
                 "effective_start",
                 F.when(F.col("rn") == 1, F.to_timestamp(F.lit("1900-01-01 00:00:00")))
                  .otherwise(F.col("__START_AT"))
             )
             .withColumn(
                 "effective_end",
                 F.coalesce(F.col("__END_AT"), F.to_timestamp(F.lit("9999-12-31 23:59:59")))
             )
             .drop("rn")
    )

# =========================================================================
# 3. GOLD FACT SESSIONS (Session Aggregation + Attribution + PIT Join)
# =========================================================================

@dlt.table(
    name="gold.fct_sessions",
    comment="Fact table of user browsing sessions. Grain: one row per user session."
)
def gold_fct_sessions():
    clicks = dlt.read("silver.silver_clickstream")
    bookings = dlt.read("silver.silver_bookings")
    users = dlt.read("users_pit_staging")

    # Aggregate clickstream events to the session grain (30-min window)
    session_agg = (
        clicks.groupBy("session_id", "user_id")
              .agg(
                  F.min("event_timestamp").alias("session_start_time"),
                  F.max("event_timestamp").alias("session_end_time"),
                  F.count("event").alias("event_count"),
                  F.first("device_type", ignorenulls=True).alias("device_type"),
                  F.first("referrer", ignorenulls=True).alias("referrer"),
                  F.first("property_id", ignorenulls=True).alias("property_id"),
                  F.max(F.when(F.col("event") == "search", 1).otherwise(0)).alias("has_search"),
                  F.max(F.when(F.col("event") == "view", 1).otherwise(0)).alias("has_view"),
                  F.max(F.when(F.col("event").isin("booking_attempt", "confirm_booking"), 1).otherwise(0)).alias("has_booking_attempt")
              )
    )

    # Session-to-Booking Attribution
    booking_join = (
        session_agg.join(
            bookings.select(
                F.col("booking_id"),
                F.col("user_id").alias("b_user_id"),
                F.col("created_at").alias("booking_time")
            ),
            (session_agg["user_id"] == F.col("b_user_id")) & 
            (F.col("booking_time") >= session_agg["session_start_time"]) & 
            (F.col("booking_time") <= session_agg["session_end_time"] + F.expr("INTERVAL 30 MINUTES")),
            "left"
        ).drop("b_user_id")
    )

    with_attribution = (
        booking_join.withColumn("is_booked", F.when(F.col("booking_id").isNotNull(), 1).otherwise(0))
    )

    # Point-in-Time Join against expanded PIT intervals
    return (
        with_attribution.join(
            users,
            (with_attribution["user_id"] == users["user_id"]) &
            (with_attribution["session_start_time"] >= users["effective_start"]) &
            (with_attribution["session_start_time"] < users["effective_end"]),
            "left"
        ).select(
            with_attribution["session_id"],
            with_attribution["user_id"],
            F.coalesce(users["country"], F.lit("Unknown")).alias("user_country"),
            F.coalesce(users["user_type"], F.lit("standard")).alias("user_type"),
            with_attribution["property_id"],
            with_attribution["booking_id"],
            with_attribution["device_type"],
            with_attribution["referrer"],
            with_attribution["session_start_time"],
            with_attribution["session_end_time"],
            with_attribution["event_count"],
            with_attribution["has_search"],
            with_attribution["has_view"],
            with_attribution["has_booking_attempt"],
            with_attribution["is_booked"]
        )
    )

# =========================================================================
# 4. GOLD FACT BOOKINGS (Point-in-Time Join with Retroactive Interval)
# =========================================================================

@dlt.table(
    name="gold.fct_bookings",
    comment="Fact table of completed bookings. Grain: one row per booking transaction."
)
def gold_fct_bookings():
    bookings = dlt.read("silver.silver_bookings")
    users = dlt.read("users_pit_staging")

    return (
        bookings.join(
            users,
            (bookings["user_id"] == users["user_id"]) &
            (bookings["created_at"] >= users["effective_start"]) &
            (bookings["created_at"] < users["effective_end"]),
            "left"
        ).select(
            bookings["booking_id"],
            bookings["user_id"],
            F.coalesce(users["country"], F.lit("Unknown")).alias("user_country"),
            bookings["property_id"],
            bookings["check_in"],
            bookings["check_out"],
            bookings["guests_count"],
            bookings["total_amount"],
            bookings["booking_status"],
            bookings["created_at"].alias("booking_timestamp")
        )
    )

# =========================================================================
# 5. GOLD FACT REVIEWS (Point-in-Time Join with Retroactive Interval)
# =========================================================================

@dlt.table(
    name="gold.fct_reviews",
    comment="Fact table of property reviews and ratings. Grain: one row per submitted review."
)
def gold_fct_reviews():
    reviews = dlt.read("silver.silver_reviews")
    users = dlt.read("users_pit_staging")

    return (
        reviews.join(
            users,
            (reviews["user_id"] == users["user_id"]) &
            (reviews["created_at"] >= users["effective_start"]) &
            (reviews["created_at"] < users["effective_end"]),
            "left"
        ).select(
            reviews["review_id"],
            reviews["booking_id"],
            reviews["user_id"],
            F.coalesce(users["country"], F.lit("Unknown")).alias("user_country"),
            reviews["property_id"],
            reviews["rating"],
            reviews["review_comment"],
            reviews["is_deleted"],
            reviews["created_at"].alias("review_timestamp")
        )
    )